# Лабораторная работа 1. Предварительный анализ данных

**Выполнил:** Моисеев Г. В., группа 4414

**Вариант 16.** Набор данных `drivers.csv`. В методичке 15 вариантов, поэтому номер взят по кругу: 16 → 1, задания варианта 1.

**Цель работы:** осуществить предварительную обработку данных csv-файла, выявить и устранить проблемы в этих данных.

## 1. Загрузка данных

Подключаю pandas и читаю файл. Вывод всех столбцов включён, чтобы широкие таблицы не обрезались. NumPy нужен для поиска бесконечных значений, модуль `abc` — для базового класса преобразований.

In [ ]:
from abc import ABC, abstractmethod

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

DATA_PATH = 'data/drivers.csv'

df = pd.read_csv(DATA_PATH)

## 2. Первые 20 строк

In [2]:
df.head(20)

,START_DATE*,END_DATE*,CATEGORY*,START*,STOP*,MILES*,PURPOSE*,time,speed,price
0,2016-01-01 21:11:00,2016-01-01 21:17:00,Business,Fort Pierce,Fort Pierce,5.1,Meal/Entertain,6.0,51.000000,788.0
1,2016-01-02 01:25:00,2016-01-02 01:37:00,Business,Fort Pierce,Fort Pierce,5.0,NaN,12.0,25.000000,1237.0
2,2016-01-02 20:25:00,2016-01-02 20:38:00,Business,Fort Pierce,Fort Pierce,4.8,Errand/Supplies,13.0,22.153846,1312.0
3,2016-01-05 17:31:00,2016-01-05 17:45:00,Business,Fort Pierce,Fort Pierce,4.7,Meeting,14.0,20.142857,1387.0
4,2016-01-06 14:42:00,2016-01-06 15:49:00,Business,Fort Pierce,West Palm Beach,63.7,Customer Visit,67.0,57.044776,5376.0
5,2016-01-06 17:15:00,2016-01-06 17:19:00,Business,West Palm Beach,West Palm Beach,4.3,Meal/Entertain,4.0,64.500000,638.0
6,2016-01-06 17:30:00,2016-01-06 17:35:00,Business,West Palm Beach,Palm Beach,7.1,Meeting,5.0,85.200000,714.0
7,2016-01-07 13:27:00,2016-01-07 13:33:00,Business,Cary,Cary,0.8,Meeting,6.0,8.000000,787.0
8,2016-01-10 08:05:00,2016-01-10 08:25:00,Business,Cary,Morrisville,8.3,Meeting,20.0,24.900000,1838.0
9,2016-01-10 12:17:00,2016-01-10 12:44:00,Business,Jamaica,New York,16.5,Customer Visit,27.0,36.666667,2365.0


## 3. Обзор данных

Набор данных содержит информацию о поездках в такси. Каждая строка — одна поездка.

| Столбец | Что хранит |
|---|---|
| `START_DATE*` | дата и время начала поездки |
| `END_DATE*` | дата и время окончания поездки |
| `CATEGORY*` | категория поездки: `Business` (деловая) или `Personal` (личная) |
| `START*` | место начала поездки |
| `STOP*` | место окончания поездки |
| `MILES*` | пройденное расстояние, мили |
| `PURPOSE*` | цель поездки |
| `time` | длительность поездки, минуты |
| `speed` | средняя скорость, мили в час |
| `price` | стоимость поездки |

Последних трёх столбцов нет в описании варианта. Что в них лежит, проверил по значениям: `time` равен разнице между окончанием и началом поездки в минутах, а `speed` — это `MILES*` / `time` · 60.

In [3]:
duration = pd.to_datetime(df['END_DATE*']) - pd.to_datetime(df['START_DATE*'])
moving = df['time'] > 0

print('time совпадает с длительностью:', (duration.dt.total_seconds() / 60 == df['time']).all())
print('speed = MILES* / time * 60:', ((df['MILES*'] / df['time'] * 60 - df['speed']).abs() < 1e-9)[moving].all())

time совпадает с длительностью: True
speed = MILES* / time * 60: True


## 4. Общая информация о данных

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1099 entries, 0 to 1098
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   START_DATE*  1099 non-null   object 
 1   END_DATE*    1099 non-null   object 
 2   CATEGORY*    1099 non-null   object 
 3   START*       1099 non-null   object 
 4   STOP*        1099 non-null   object 
 5   MILES*       1099 non-null   float64
 6   PURPOSE*     598 non-null    object 
 7   time         1099 non-null   float64
 8   speed        1099 non-null   float64
 9   price        1099 non-null   float64
dtypes: float64(4), object(6)
memory usage: 86.0+ KB


В таблице 1099 строк и 10 столбцов. Что видно сразу:

- пропуски есть только в `PURPOSE*`: заполнено 598 значений из 1099;
- даты начала и окончания прочитались как `object`, то есть как строки;
- `time` и `price` хранятся как `float64`, хотя по первым строкам значения в них целые.

## 5. Числовые столбцы

Перед `describe` проверяю числовые столбцы на бесконечные значения: `speed` получен делением, и при нулевой длительности в нём может оказаться `inf`, из-за которого среднее и стандартное отклонение не считаются.

In [5]:
np.isinf(df.select_dtypes('number')).sum()

MILES*    0
time      0
speed     4
price     0
dtype: int64

В `speed` четыре бесконечных значения. Смотрю эти поездки:

In [6]:
df[np.isinf(df['speed'])]

,START_DATE*,END_DATE*,CATEGORY*,START*,STOP*,MILES*,PURPOSE*,time,speed,price
750,2016-09-06 17:49:00,2016-09-06 17:49:00,Business,Unknown Location,Unknown Location,69.1,NaN,0.0,inf,359.0
760,2016-09-16 07:08:00,2016-09-16 07:08:00,Business,Unknown Location,Unknown Location,1.6,NaN,0.0,inf,338.0
797,2016-10-08 15:03:00,2016-10-08 15:03:00,Business,Karachi,Karachi,3.6,NaN,0.0,inf,338.0
806,2016-10-13 13:02:00,2016-10-13 13:02:00,Business,Islamabad,Islamabad,0.7,NaN,0.0,inf,337.0


У всех четырёх время начала и окончания совпадает, `time` = 0, поэтому скорость при делении на ноль стала `inf`. Длительность у этих поездок записана с ошибкой: за 0 минут нельзя проехать 69.1 мили. Скорость для них посчитать нельзя, поэтому заменяю `inf` на `NaN` — значение неизвестно. Сами строки обработаю в разделе 7 вместе с остальными пропусками.

Все преобразования таблицы оформлены отдельными классами с общим методом `apply`: он принимает датафрейм и возвращает новый, исходный не меняется.

In [7]:
class Transformation(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        ...


class InfinityReplacer(Transformation):
    def __init__(self, columns: list[str]):
        self._columns = columns

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        replaced = {column: df[column].replace(np.inf, np.nan)
                    for column in self._columns}
        return df.assign(**replaced)

In [8]:
df = InfinityReplacer(['speed']).apply(df)
df.describe()

,MILES*,time,speed,price
count,1099.000000,1099.000000,1095.000000,1099.000000
mean,10.803094,23.300273,27.379427,2085.929936
std,22.044580,27.745836,44.159739,2084.368402
min,0.500000,0.000000,3.917355,337.000000
25%,2.900000,10.000000,15.333333,1087.000000
50%,6.000000,16.000000,21.333333,1539.000000
75%,10.500000,27.000000,28.875000,2365.500000
max,310.300000,336.000000,906.000000,25569.000000


- `MILES*` — от 0.5 до 310.3 миль. Среднее 10.8, а медиана 6.0: большинство поездок короткие, а несколько дальних поднимают среднее.
- `time` — от 0 до 336 минут, медиана 16 минут.
- `speed` — посчитан по 1095 поездкам, медиана 21.3 мили в час. Максимум 906 миль в час нереален: `time` округлён до минут, и у поездок длиной 1–3 минуты скорость сильно завышается. В заданиях варианта `speed` не используется, поэтому эти значения не исправляю.
- `price` — от 337 до 25 569, медиана 1539.

## 6. Названия столбцов

In [9]:
df.columns

Index(['START_DATE*', 'END_DATE*', 'CATEGORY*', 'START*', 'STOP*', 'MILES*',
       'PURPOSE*', 'time', 'speed', 'price'],
      dtype='object')

У семи столбцов в конце названия стоит `*`. Её приходится писать при каждом обращении к столбцу, и её легко пропустить. Звёздочку убираю. Регистр не трогаю, чтобы названия совпадали с теми, что используются в заданиях: `CATEGORY`, `START`, `MILES`. Цель поездки в заданиях называется `PURPOSEroute`, в файле это столбец `PURPOSE*`, после переименования — `PURPOSE`.

In [10]:
class ColumnRenamer(Transformation):
    def __init__(self, mapping: dict[str, str]):
        self._mapping = mapping

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.rename(columns=self._mapping)

In [11]:
renamer = ColumnRenamer({column: column.rstrip('*') for column in df.columns})
df = renamer.apply(df)
df.columns

Index(['START_DATE', 'END_DATE', 'CATEGORY', 'START', 'STOP', 'MILES',
       'PURPOSE', 'time', 'speed', 'price'],
      dtype='object')

## 7. Пропуски

Для проверок данных (пропуски, дубликаты, уникальные значения) заведён класс `DataInspector`. Он только читает таблицу и ничего в ней не меняет.

In [12]:
class DataInspector:
    def __init__(self, df: pd.DataFrame):
        self._df = df

    def missing_counts(self) -> pd.Series:
        return self._df.isna().sum()

    def duplicate_count(self) -> int:
        return int(self._df.duplicated().sum())

    def unique_values(self, column: str) -> list:
        return self._df[column].unique().tolist()

In [13]:
DataInspector(df).missing_counts()

START_DATE      0
END_DATE        0
CATEGORY        0
START           0
STOP            0
MILES           0
PURPOSE       501
time            0
speed           4
price           0
dtype: int64

Пропуски есть в двух столбцах.

`PURPOSE` — 501 пропуск, это 46 % строк. Если удалить такие строки, пропадёт почти половина поездок, хотя остальные столбцы в них заполнены. Цель поездки — категориальный признак, поэтому пропуски заменяю значением «не определена». Так же пропуски подписаны в примере результата к заданию 1.

`speed` — 4 пропуска, это поездки с нулевой длительностью из раздела 5. Подставить вместо скорости нечего: и `speed`, и `time` у них ошибочные. Строк всего четыре (0.4 % данных), поэтому их удаляю.

In [14]:
class MissingValueFiller(Transformation):
    def __init__(self, column: str, value):
        self._column = column
        self._value = value

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.assign(**{self._column: df[self._column].fillna(self._value)})


class MissingRowsDropper(Transformation):
    def __init__(self, columns: list[str]):
        self._columns = columns

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna(subset=self._columns).reset_index(drop=True)

In [15]:
df = MissingValueFiller('PURPOSE', 'не определена').apply(df)
df = MissingRowsDropper(['speed']).apply(df)
DataInspector(df).missing_counts()

START_DATE    0
END_DATE      0
CATEGORY      0
START         0
STOP          0
MILES         0
PURPOSE       0
time          0
speed         0
price         0
dtype: int64

Пропусков больше нет, в таблице осталось 1095 строк.

## 8. Дубликаты

Явные дубликаты — строки, которые полностью повторяют уже встречавшиеся:

In [16]:
inspector = DataInspector(df)
inspector.duplicate_count()

0

Явных дубликатов нет, удалять нечего.

Неявные дубликаты — разные написания одного значения. Смотрю уникальные значения в категориальных столбцах `CATEGORY` и `PURPOSE`. `START` и `STOP` по заданию можно не проверять.

In [17]:
for column in ['CATEGORY', 'PURPOSE']:
    print(column)
    for value in inspector.unique_values(column):
        print('   ', value)

CATEGORY
    Business
    Personal
PURPOSE
    Meal/Entertain
    не определена
    Errand/Supplies
    Meeting
    Customer Visit
    Temporary Site
    Between Offices
    Charity ($)
    Commute
    Moving
    Airport/Travel


В `CATEGORY` два значения, в `PURPOSE` — десять целей и «не определена». Разных написаний одного и того же значения нет, неявных дубликатов тоже нет.

## 9. Типы данных

- `START_DATE` и `END_DATE` — это дата и время, перевожу их в `datetime`. Формат в файле: `2016-01-01 21:11:00`, то есть `%Y-%m-%d %H:%M:%S`.
- `time` и `price` перевожу в `int64`, если в них действительно только целые значения. Проверяю:

In [18]:
(df[['time', 'price']] % 1 == 0).all()

time     True
price    True
dtype: bool

In [19]:
class DatetimeConverter(Transformation):
    def __init__(self, columns: list[str], date_format: str):
        self._columns = columns
        self._format = date_format

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        converted = {column: pd.to_datetime(df[column], format=self._format)
                     for column in self._columns}
        return df.assign(**converted)


class TypeConverter(Transformation):
    def __init__(self, dtypes: dict[str, str]):
        self._dtypes = dtypes

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.astype(self._dtypes)

In [20]:
df = DatetimeConverter(['START_DATE', 'END_DATE'], '%Y-%m-%d %H:%M:%S').apply(df)
df = TypeConverter({'time': 'int64', 'price': 'int64'}).apply(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   START_DATE  1095 non-null   datetime64[ns]
 1   END_DATE    1095 non-null   datetime64[ns]
 2   CATEGORY    1095 non-null   object        
 3   START       1095 non-null   object        
 4   STOP        1095 non-null   object        
 5   MILES       1095 non-null   float64       
 6   PURPOSE     1095 non-null   object        
 7   time        1095 non-null   int64         
 8   speed       1095 non-null   float64       
 9   price       1095 non-null   int64         
dtypes: datetime64[ns](2), float64(2), int64(2), object(4)
memory usage: 85.7+ KB


Даты теперь имеют тип `datetime64`, `time` и `price` — `int64`. Поездки в наборе данных — с 1 января по 23 декабря 2016 года:

In [21]:
df['START_DATE'].min(), df['START_DATE'].max()

(Timestamp('2016-01-01 21:11:00'), Timestamp('2016-12-23 11:33:00'))

## 10. Группировки и сводные таблицы

Все четыре задания собраны в классе `TripsAnalysis`: каждый метод возвращает результат одного задания.

In [22]:
class TripsAnalysis:
    def __init__(self, trips: pd.DataFrame):
        self._trips = trips

    def purpose_counts_by_category(self) -> pd.Series:
        return self._trips.groupby(['CATEGORY', 'PURPOSE'])['PURPOSE'].count()

    def start_counts_by_category(self) -> pd.DataFrame:
        counts = self._trips.groupby(['CATEGORY', 'START'])[['START']].count()
        return counts.rename(columns={'START': 'count'}).sort_values('count')

    def mean_miles_by_purpose(self) -> pd.DataFrame:
        pivot = self._trips.pivot_table(index='PURPOSE', values='MILES',
                                        aggfunc='mean')
        return pivot.sort_values('MILES', ascending=False).round(2)

    def mean_miles_by_start_and_purpose(self) -> pd.DataFrame:
        pivot = self._trips.pivot_table(
            index='START', columns=['CATEGORY', 'PURPOSE'], values='MILES',
            aggfunc='mean'
        )
        return pivot.sort_index(ascending=False).round(2)


analysis = TripsAnalysis(df)

### Задание 1

Группировка по `CATEGORY` и количество поездок каждого типа (по цели маршрута).

In [23]:
analysis.purpose_counts_by_category()

CATEGORY  PURPOSE        
Business  Airport/Travel       2
          Between Offices     17
          Customer Visit      94
          Errand/Supplies    110
          Meal/Entertain     145
          Meeting            178
          Temporary Site      46
          не определена      426
Personal  Charity ($)          1
          Commute              1
          Moving               4
          не определена       71
Name: PURPOSE, dtype: int64

Деловых поездок 1018, личных — 77. У деловых чаще всего указана цель Meeting (178), Meal/Entertain (145) и Errand/Supplies (110). У 426 деловых поездок цель не указана. У личных поездок цель почти никогда не указывают: 71 из 77, а у остальных шести это Moving (4), Commute (1) и Charity ($) (1).

### Задание 2

Группировка по `CATEGORY` и количество поездок для каждой точки старта `START`. Результат — датафрейм, столбец с количеством называется `count`, сортировка по возрастанию `count`.

In [24]:
start_counts = analysis.start_counts_by_category()
start_counts

count
CATEGORY START                           
Business Almond                         1
         Arabi                          1
         Arlington Park at Amberly      1
         Arlington                      1
         Austin                         1
...                                   ...
         Islamabad                     51
         Whitebridge                   59
         Morrisville                   79
         Unknown Location             133
         Cary                         198

[199 rows x 1 columns]

В таблице 199 пар «категория — точка старта». Больше всего поездок начинается в Cary (198 деловых), дальше идут Unknown Location (133) и Morrisville (79). Unknown Location — это поездки, у которых место старта не записано. В начале таблицы — места, из которых была всего одна поездка.

### Задание 3

Сводная таблица — среднее количество пройденных миль по каждой цели поездки. Сортировка по убыванию `MILES`, округление до двух знаков.

In [25]:
analysis.mean_miles_by_purpose()

,MILES
PURPOSE,
Commute,180.20
Customer Visit,21.90
Meeting,15.59
Charity ($),15.10
Between Offices,11.23
Temporary Site,10.01
не определена,9.68
Airport/Travel,6.20
Meal/Entertain,5.66


Первое место у Commute — 180.2 мили, но это одна поездка, так что показатель ничего не говорит о цели в целом. Среди целей с большим числом поездок самые длинные — Customer Visit (21.9 мили) и Meeting (15.59). Самые короткие — Errand/Supplies (4.14) и Meal/Entertain (5.66).

### Задание 4

Сводная таблица — среднее количество пройденных миль по каждой цели каждой категории (столбцы) и каждой точке старта (строки). Сортировка по убыванию `START`. Значения округлены с помощью `round` до двух знаков, как в задании 3.

In [26]:
analysis.mean_miles_by_start_and_purpose()

CATEGORY              Business                                                 \
PURPOSE         Airport/Travel Between Offices Customer Visit Errand/Supplies   
START                                                                           
Winston Salem              NaN             NaN            NaN             NaN   
Whitebridge                NaN             6.1           7.10            2.76   
Westpark Place             NaN             NaN            NaN            2.00   
Weston                     NaN             NaN            NaN             NaN   
West University            NaN             NaN            NaN             NaN   
...                        ...             ...            ...             ...   
Arlington                  NaN             NaN            NaN             NaN   
Arabi                      NaN             NaN            NaN             NaN   
Apex                       NaN             NaN           5.36             NaN   
Almond                     NaN             NaN            NaN             NaN   
Agnew                      NaN             NaN            NaN             NaN   

CATEGORY                                                             \
PURPOSE         Meal/Entertain Meeting Temporary Site не определена   
START                                                                 
Winston Salem              NaN  133.60            NaN           NaN   
Whitebridge               5.58    6.33            NaN          3.07   
Westpark Place            3.05     NaN            NaN          1.91   
Weston                    3.80     NaN            NaN           NaN   
West University           2.10     NaN            NaN          2.30   
...                        ...     ...            ...           ...   
Arlington                  NaN     NaN            NaN          4.90   
Arabi                    17.00     NaN            NaN           NaN   
Apex                      6.07    7.30            8.8          3.73   
Almond                     NaN     NaN            NaN         15.20   
Agnew                      NaN     NaN            NaN          2.78   

CATEGORY           Personal                               
PURPOSE         Charity ($) Commute Moving не определена  
START                                                     
Winston Salem           NaN     NaN    NaN           NaN  
Whitebridge             NaN     NaN    NaN          2.33  
Westpark Place          NaN     NaN    NaN          2.95  
Weston                  NaN     NaN    NaN          4.20  
West University         NaN     NaN    NaN           NaN  
...                     ...     ...    ...           ...  
Arlington               NaN     NaN    NaN           NaN  
Arabi                   NaN     NaN    NaN           NaN  
Apex                    NaN     NaN    NaN           NaN  
Almond                  NaN     NaN    NaN           NaN  
Agnew                   NaN     NaN    NaN           NaN  

[174 rows x 12 columns]

Таблица получилась размером 174 × 12: 174 точки старта и 12 сочетаний категории и цели. Большая часть ячеек пустая (`NaN`): из большинства мест ездили с одной-двумя целями. Больше всего заполненных ячеек у мест, откуда ездили чаще всего: у Cary и Morrisville — 9 из 12, у Unknown Location — 8.

## 11. Выводы

Набор данных `drivers.csv` содержит 1099 поездок в такси за 2016 год: даты, категорию, места начала и окончания, расстояние, цель, длительность, скорость и стоимость.

При предобработке:

- из названий семи столбцов убрана звёздочка;
- у четырёх поездок нулевая длительность и бесконечная скорость; `inf` заменён на `NaN`, эти строки удалены вместе с пропусками;
- 501 пропуск в `PURPOSE` (46 % строк) заменён на «не определена» — удаление строк стоило бы почти половины данных;
- явных дубликатов нет, неявных в `CATEGORY` и `PURPOSE` тоже нет;
- даты начала и окончания переведены в `datetime`, `time` и `price` — в `int64`.

После обработки осталось 1095 поездок.

По группировкам и сводным таблицам:

- 93 % поездок деловые (1018 из 1095); у личных поездок цель почти всегда не указана;
- основные деловые цели — встречи, питание и мелкие поручения;
- поездки в основном начинаются в Cary, Morrisville и неизвестном месте (Unknown Location);
- дальше всего ездят к клиентам (в среднем 21.9 мили) и на встречи (15.59), ближе всего — по поручениям и на обед (4–6 миль). Среднее у Commute (180.2) посчитано по одной поездке и не показательно.